# M27 — Turn Text into Tokens

**Objective:** understand tokenization by inspecting how text becomes model input.

A language model does not consume the sentence you typed. It consumes a
sequence of **token pieces** mapped to **integer IDs**, then padded or
truncated to a length. The useful whole is:

`text → normalize → pieces → IDs → [BOS]/[EOS] → pad or truncate → padding mask`

This notebook uses a **bundled** teaching tokenizer
(`v06-teaching-tokenizer`, version `v06.1`). Nothing is downloaded.
Embeddings, attention, and transformer blocks stay closed (M28-M30).


## Working contract

Every experiment follows **predict → act → observe → explain**. **Predict before running**
each action cell and timestamp the prediction in your own evidence log.
A prediction is falsifiable: a piece list, a count, or an ID.

Do not download a Hugging Face model, do not call tiktoken, and do not
treat token IDs as meanings. If a failure can be diagnosed from a token
count and a dropped suffix, stay there.

The repository does not prefill learner answers, ADR text, or competence.


In [ ]:
from pathlib import Path
import inspect
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT = None
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "missions" / "M27" / "tokenization_core.py").is_file():
        ROOT = candidate
        break
if ROOT is None:
    raise RuntimeError("Run from the LearningOS-AI repository or its labs directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from missions.M27.tokenization_core import (
    TokenBudgetError,
    compare_schemes,
    encoding_report,
    load_texts,
    load_tokenizer,
    normalize_text,
    pack_for_context,
    pretokens,
)

word = load_tokenizer(scheme="word")
bpe = load_tokenizer(scheme="bpe")
TEXTS = load_texts()

print("repository root:", ROOT)
print("word identity:", word.identity)
print("bpe identity:", bpe.identity)
print("canonical sentence:", TEXTS["canonical_sentence"])
print("tokenizer version:", bpe.version)
print("downloaded:", bpe.identity.downloaded)
print("word vocab size:", word.identity.vocab_size)
print("bpe vocab size:", bpe.identity.vocab_size)


## M03 boundary: Python in, transformers out

M03 already lets you manipulate strings and lists. M27 asks what a
**model input unit** is. It is not a word, not a character always, and
not yet a vector.

What this mission **opens:** normalization, token pieces, integer IDs,
word versus tiny BPE segmentation, special tokens, padding, truncation,
sequence length, padding masks, and **token budget** reasoning.

What stays **deferred:**
- M28 — semantic embeddings and similarity search
- M29 — attention and using a mask inside a score matrix
- M30 — a transformer block

A padding mask here only answers: *is this position a real token or
`[PAD]`?* Production libraries sometimes call that vector
`attention_mask`. The name is not this mission. Do not compute scores.


## Frozen teaching fixtures

Declare the useful whole **before** the first encoding.

| Fixture | Value |
| --- | --- |
| Family | `v06-teaching-tokenizer` |
| Version | `v06.1` |
| Schemes | `word` (lookup) and `bpe` (tiny frozen merges) |
| Specials | `[PAD]=0`, `[UNK]=1`, `[BOS]=2`, `[EOS]=3` |
| Normalization | lowercase, strip, collapse whitespace |
| Truncation / padding | right |
| Canonical sentence | `the cat sat on the mat` |
| Download | false — JSON in `datasets/M27/` |

Primary sources: `hf-llm-course` and `karpathy-zero-to-hero` in
`data/source_registry.json`. Skip production encodings here.

The BPE alphabet is the teaching ASCII set (letters, digits, a few
marks, plus `▁` for a word start). For this fixture those symbols match
UTF-8 bytes. That is enough to see subword / byte-like segmentation
without fetching a 50k vocabulary.


In [ ]:
print("specials", bpe.identity.special_tokens)
print("pad/unk/bos/eos ids", bpe.pad_id, bpe.unk_id, bpe.bos_id, bpe.eos_id)
print("truncation_side", bpe.truncation_side, "padding_side", bpe.padding_side)
print("first 12 word vocab", word.vocab[:12])
print("bpe merges (first 8)", bpe.merges[:8])
print("canonical pretokens", pretokens(TEXTS["canonical_sentence"]))
print("normalized example", normalize_text("  The   CAT sat.  "))
assert bpe.version == "v06.1"
assert word.token_to_id("[PAD]") == 0
assert bpe.identity.downloaded is False


### Identity is part of the contract

Every encoding below carries `tokenizer_name` and `tokenizer_version`.
If either changes, IDs are not comparable. That is the migration trigger
M28 will inherit. Vocabulary membership is not understanding: `cat` has
an ID because it was in the teaching corpus, not because the tokenizer
knows about animals.


## Predict before running — surface variation

Timestamp a prediction before `run-surface`.

Start from `The cat sat on the mat.` Change **one** surface feature at
a time. The sentence intent stays the same.

Predict token pieces and counts for:
- extra internal whitespace (`The  cat sat on the mat.`)
- casing (`THE CAT SAT ON THE MAT.`)
- punctuation (`The cat sat on the mat!`)

Will extra spaces become extra tokens? Will `!` and `.` be the same ID?


In [ ]:
variants = TEXTS["surface_variants"]
surface_rows = []
for name, text in variants.items():
    encoded = bpe.encode(text)
    surface_rows.append((name, encoded.normalized, encoded.tokens, encoded.length))
    print(name, encoding_report(encoded))

base_ids = bpe.encode(variants["base"]).ids
assert base_ids == bpe.encode(variants["whitespace"]).ids
assert base_ids == bpe.encode(variants["casing"]).ids
assert base_ids != bpe.encode(variants["punctuation"]).ids
print("normalization hid casing and extra spaces; punctuation changed a piece")


### Normalization is a policy, not a no-op

Casing and repeated spaces disappeared before segmentation, so the IDs
matched. The exclamation mark did not: it is a different piece from
`.`. The invariant was sentence intent, not byte-for-byte source text.
If you predicted that extra spaces would add tokens, the observation
falsifies that under **this** tokenizer's policy.


## Predict before running — rare strings

Timestamp a prediction before `run-rare`.

Keep the tokenizer and vocabulary fixed. Introduce one rare surface at
a time from `TEXTS["rare_strings"]`:

- identifier `xgztq9`
- a URL-like string
- number `99281`

Predict, for **word** and **bpe**:
- whether the rare span is one piece, many pieces, or `[UNK]`
- which scheme produces the longer sequence

Do not claim that more pieces means more meaning.


In [ ]:
for name, text in TEXTS["rare_strings"].items():
    compared = compare_schemes(text)
    print(name)
    print("  word", compared["word_length"], compared["word_tokens"])
    print("  bpe ", compared["bpe_length"], compared["bpe_tokens"])
    print("  delta", compared["length_delta"])

rare = compare_schemes(TEXTS["rare_strings"]["identifier"])
assert "[UNK]" in rare["word_tokens"]
assert rare["bpe_length"] > rare["word_length"]
print("rare identifier: word UNK vs BPE character pieces")


### Vocabulary membership is not semantics

Frame sentences keep in-vocab words (`the ticket … sat`, `pay invoice … now`).
The only word-scheme `[UNK]` in the identifier example is `xgztq9`; `open`
and `due` are no longer extra OOVs. BPE still has the characters, so it
emits a longer chain. The URL-like string splits on punctuation. None of
this retrieves similar tickets. That is M28.


## Predict before running — pieces to IDs and back

Timestamp a prediction before `run-ids`.

For the canonical sentence `the cat sat on the mat`, predict:

- the BPE piece list including `[BOS]` and `[EOS]`
- whether decode returns the **normalized** sentence
- whether `the` has the same ID both times it appears

If you predict eight tokens, write that down so eight (or not) can
falsify it.


In [ ]:
canonical = TEXTS["canonical_sentence"]
word_enc = word.encode(canonical)
bpe_enc = bpe.encode(canonical)
print("word", encoding_report(word_enc))
print("bpe ", encoding_report(bpe_enc))
print("word decode", word.decode(word_enc.ids))
print("bpe decode", bpe.decode(bpe_enc.ids))
print("id of first the", bpe_enc.ids[1], "id of second the", bpe_enc.ids[5])
print("token for that id", bpe.id_to_token(bpe_enc.ids[1]))
assert word_enc.tokens == tuple(TEXTS["expected"]["canonical_word_tokens"])
assert bpe_enc.tokens == tuple(TEXTS["expected"]["canonical_bpe_tokens"])
assert bpe.decode(bpe_enc.ids) == canonical
assert bpe_enc.ids[1] == bpe_enc.ids[5]
print("same piece, same ID; decode matches normalized text")


### An ID is an index

`the` maps to the same integer wherever it occurs because the
vocabulary is a lookup table. Decode rebuilds the normalized string,
not the original casing. That is the whole representation this mission
owns. A later mission may place a vector at each ID. It is not allowed
to pretend the ID already *is* that vector.


## Predict before running — special tokens and effective length

Timestamp a prediction before `run-specials`.

For `the cat sat`, predict:

- content length with `add_special_tokens=False`
- effective length with `[BOS]` and `[EOS]`
- what `max_length=4` with specials and truncation must keep

Special tokens are not sentence words. They still spend the context
budget.


In [ ]:
sample = "the cat sat"
plain = bpe.encode(sample, add_special_tokens=False)
wrapped = bpe.encode(sample, add_special_tokens=True)
short = bpe.encode(sample, add_special_tokens=True, max_length=4, truncation=True)
print("plain", plain.tokens, "len", plain.length)
print("wrapped", wrapped.tokens, "len", wrapped.length)
print("max_length=4", short.tokens, "dropped", short.dropped_tokens)
assert wrapped.length == plain.length + 2
assert short.tokens[0] == "[BOS]" and short.tokens[-1] == "[EOS]"
assert short.truncated
print("specials reserved two slots; content was truncated to fit")


### Boundaries are tokens

`[BOS]` and `[EOS]` made a 3-piece sentence occupy 5 IDs. With
`max_length=4` the implementation keeps the specials and drops content
from the **right**, so the surviving window is not "first four words."
Count specials or you will lie about context.


## Predict before running — padding a batch

Timestamp a prediction before `run-padding`.

Batch the three strings in `TEXTS["padding_batch"]` to `max_length=10`.
Predict:

- which row needs the most `[PAD]`
- the padding mask pattern (1 = real, 0 = pad)
- whether the non-pad prefix of each row matches encoding that row alone

The original token **order** must stay fixed. Padding is on the right.


In [ ]:
batch = bpe.encode_batch(
    TEXTS["padding_batch"],
    add_special_tokens=True,
    max_length=10,
    truncation=True,
    padding=True,
)
print("tokenizer", batch.tokenizer_name, batch.tokenizer_version, "scheme", batch.scheme)
for text, encoded in zip(TEXTS["padding_batch"], batch.encodings):
    print(text)
    print("  ids ", encoded.ids)
    print("  mask", encoded.padding_mask)
    print("  tokens", encoded.tokens)
    alone = bpe.encode(text, add_special_tokens=True)
    assert encoded.ids[: alone.length] == alone.ids
    assert encoded.padding_mask[: alone.length] == tuple(1 for _ in alone.ids)
print("pad id is", bpe.pad_id, "and never appears in a non-pad prefix")


### A mask is metadata

`[PAD]` IDs fill the rectangular batch. The padding mask marks them so
a later stage can ignore those slots. This notebook does not multiply
that mask into scores. Order of the real tokens is the evidence that
padding did not scramble the sentence.


## Predict before running — a fixed context limit

Timestamp a prediction before `run-truncation`.

Text: `please inspect ticket 4412 then process the invoice now`.

Encode with specials. Then encode again with `max_length=8` and
truncation on. Predict **exactly** which content pieces disappear.
The tokenizer and the text stay fixed; only the length cap changes.


In [ ]:
trunc_text = TEXTS["truncation_text"]
full = bpe.encode(trunc_text, add_special_tokens=True)
cut = bpe.encode(trunc_text, add_special_tokens=True, max_length=8, truncation=True)
print("full len", full.length, full.tokens)
print("cut  len", cut.length, cut.tokens)
print("dropped", cut.dropped_tokens)
print("decoded kept", bpe.decode(cut.ids))
print("decoded dropped", bpe.decode_pieces(cut.dropped_tokens))
assert cut.truncated
assert "invoice" not in bpe.decode(cut.ids)
assert "invoice" in bpe.decode_pieces(cut.dropped_tokens)
print("right-truncation removed the invoice tail")


### The suffix is the casualty

Right-truncation kept the ticket identifier and dropped `the invoice
now`. If the operationally important span lives at the end of a prompt,
a length cap can delete it while still returning a valid-looking ID
list. The next failure makes that operational.


## Predict before running — two local schemes, same corpus

Timestamp a prediction before `run-comparison`.

Apply **word** and **bpe** to `TEXTS["comparison_corpus"]` with the same
special-token policy. Predict:

- which sentences have equal length under both schemes
- which sentences BPE lengthens
- that neither scheme is a universal winner

Invariant: same texts, same counting policy (specials included).


In [ ]:
comparison_rows = []
for text in TEXTS["comparison_corpus"]:
    row = compare_schemes(text)
    comparison_rows.append(row)
    print(row["normalized"], "word", row["word_length"], "bpe", row["bpe_length"], "delta", row["length_delta"])

assert comparison_rows[0]["word_length"] == comparison_rows[0]["bpe_length"]
assert comparison_rows[-1]["bpe_length"] > comparison_rows[-1]["word_length"]
print("in-vocab teaching sentences can match; rare strings do not")


In [ ]:
labels = [row["normalized"][:28] for row in comparison_rows]
word_lengths = [row["word_length"] for row in comparison_rows]
bpe_lengths = [row["bpe_length"] for row in comparison_rows]
positions = range(len(labels))
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar([p - 0.2 for p in positions], word_lengths, width=0.4, label="word")
ax.bar([p + 0.2 for p in positions], bpe_lengths, width=0.4, label="bpe")
ax.set_xticks(list(positions), labels, rotation=20, ha="right")
ax.set_ylabel("tokens including specials")
ax.set_title("Same corpus, two local tokenizers")
ax.legend()
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()
print("word lengths", word_lengths)
print("bpe lengths", bpe_lengths)


### No universal winner

On the teaching sentence, both schemes emit eight tokens. On the
URL-like string, BPE is much longer. Word lookup is shorter only
because it collapses unknowns into `[UNK]`. Pick a scheme for a reason
(model-like segmentation vs human words), not because one bar is
prettier. Production systems freeze **one** identity and version.


## Code reading — normalize, encode, mask, identity

Read `TeachingTokenizer.encode`, `pack_for_context`, and `apply_bpe` in
`missions/M27/tokenization_core.py` (see also `missions/M27/code_reading.md`).
Before the next cell, predict:

1. what `max_length=4` with specials keeps
2. whether extra spaces survive `normalize_text`
3. whether a word budget can claim `heuristic_fit=True` while BPE still truncates

Do not search the file for an embedding matrix.


In [ ]:
encode_src = inspect.getsource(bpe.encode)
pack_src = inspect.getsource(pack_for_context)
for marker in ("normalize_text", "tokenize", "[BOS]", "padding_mask", "truncated"):
    print(f"encode contains {marker!r}:", marker in encode_src)
print("pack_for_context mentions tokens:", "tokens" in pack_src)
print("pack_for_context mentions words:", "words" in pack_src)
print("identity", bpe.name, bpe.version, "downloaded", bpe.identity.downloaded)
print("trace sat", bpe.trace_bpe_word("sat"))
print("spaces collapsed?", normalize_text("the    cat") == "the cat")
short = bpe.encode("the cat sat", add_special_tokens=True, max_length=4, truncation=True)
print("max_length=4 tokens", short.tokens)


## Predict before running — Controlled failure: word budget

Timestamp a prediction before `run-failure`.

Text: `please inspect ticket 4412 then approve_refund`.
Limit: `max_tokens=12`. Defect: `budget_unit="words"`.
The same text under `budget_unit="characters"` is the same class of mistake.

Predict:
- whitespace word count (does the heuristic say it fits?)
- character count versus a 48-character window
- BPE length including specials
- whether `approve_refund` survives decode after packing

The tokenizer, the text, and the token limit stay fixed. Only the
budget unit is wrong.


In [ ]:
failure = TEXTS["controlled_failure"]
print("text", failure["text"])
print("words", len(failure["text"].split()), "chars", len(failure["text"]))
print("bpe tokens needed", bpe.token_count(failure["text"]))
print("word tokens needed", word.token_count(failure["text"]))
defective = pack_for_context(
    failure["text"],
    bpe,
    max_tokens=failure["max_tokens"],
    budget_unit="words",
)
print("heuristic_fit", defective.heuristic_fit)
print("silent", defective.silent)
char_defective = pack_for_context(
    failure["text"],
    bpe,
    max_tokens=failure["max_tokens"],
    budget_unit="characters",
)
print("character heuristic_fit", char_defective.heuristic_fit)
print("character silent", char_defective.silent)
assert char_defective.silent
print("truncated", defective.truncated)
print("decoded", defective.decode(bpe))
print("dropped", defective.dropped_text, defective.dropped_tokens)
print("suffix present", defective.contains(failure["critical_suffix"], bpe))
assert defective.silent
assert not defective.contains(failure["critical_suffix"], bpe)
print("word budget claimed fit; BPE truncation ate the suffix")


### Diagnose before repair

Symptom: the packed IDs look complete, but `approve_refund` is gone.
The word count was 6, so a 1-word-1-token heuristic accepted the
string. BPE needed 22 IDs. Right-truncation to 12 then kept
`then appr` and dropped `ove_refund`.

This is a **token-budget** defect, not a reason to switch schemes or to
open M28. Characters with a "4 characters per token" rule make the
same class of mistake.


## Predict before running — smallest repair

Timestamp a prediction before `run-failure-repair`.

Predict that `budget_unit="tokens"`:

- reports overflow at `max_tokens=12` (`silent` is false)
- raises `TokenBudgetError` when `on_overflow="raise"`
- keeps `approve_refund` when `max_tokens` equals the measured BPE length

Do not repair this by removing specials or by changing the tokenizer.


In [ ]:
failure = TEXTS["controlled_failure"]
honest = pack_for_context(
    failure["text"],
    bpe,
    max_tokens=failure["max_tokens"],
    budget_unit="tokens",
)
print("honest silent", honest.silent, "truncated", honest.truncated)
print("needed", honest.original_token_count, "dropped", honest.dropped_text)
try:
    pack_for_context(
        failure["text"],
        bpe,
        max_tokens=failure["max_tokens"],
        budget_unit="tokens",
        on_overflow="raise",
    )
    raise AssertionError("expected TokenBudgetError")
except TokenBudgetError as exc:
    print("raised", exc)

repaired = pack_for_context(
    failure["text"],
    bpe,
    max_tokens=honest.original_token_count,
    budget_unit="tokens",
)
print("repaired decode", repaired.decode(bpe))
print("repaired truncated", repaired.truncated)
assert not honest.silent
assert repaired.contains(failure["critical_suffix"], bpe)
assert not repaired.truncated
print("token budget is the repair; enough tokens keep the suffix")


### Count what the model will count

The smallest repair uses the same tokenizer that will consume the
text. Overflow becomes visible. Keeping a critical instruction is a
**larger** budget, not a different meaning space. Log tokenizer name,
version, token length, and whether truncation fired — those are
migration triggers, not decorations.


## Evidence contract

Submit, in your own log (not in this repository):

- timestamped **Predict before running** notes
- surface-variation outcomes (spaces vs punctuation)
- rare-string word UNK vs BPE pieces
- canonical IDs and the decode round-trip
- padded batch with masks
- named truncated suffix
- scheme comparison without a universal winner
- word-budget diagnosis and token-budget repair

See `missions/M27/evidence_contract.yaml`. Do not paste filled evidence
into the committed notebook.


## No-AI gate

Close this notebook and complete `missions/M27/no_ai_gate.md` from a blank
page without AI-generated code, calculations, prose, or diagrams.

**Status:** [UNFILLED BY LEARNER]


## Unfilled ADR

Use `missions/M27/adr_prompt.md` to choose a V06 **teaching**
tokenizer/version/context-budget policy. Do not claim a production
encoding.

- **Status:** [UNFILLED BY LEARNER]
- **Date:** [UNFILLED BY LEARNER]
- **Owner:** [UNFILLED BY LEARNER]
- **Decision:** [UNFILLED BY LEARNER]

This notebook is not that ADR.


## M03 → M27 → M28 handoff

M03 supplies Python. M27 freezes text as token IDs with an honest
**token budget**. M28 may attach vectors to those IDs **only after**
pieces, specials, padding, truncation, and the repaired budget rule
are defended.

M29 may read the padding mask as keep/drop metadata. It must not treat
that mask as an explanation of intent. M30 still owns the block.

Reusable fixtures: `datasets/M27/teaching_tokenizer.json` and
`datasets/M27/texts.json`.


## Mission summary prompt

In your own words, using only observations from this lab:

1. Why are model input units tokens rather than human words?
2. Why did extra spaces not change IDs while `!` did?
3. How did a 6-word string lose `approve_refund` inside a 12-token window?
4. What must M28 receive that a character count cannot provide?

Leave the answers in your evidence log, not in this file.


In [ ]:
canonical = bpe.encode(TEXTS["canonical_sentence"])
assert canonical.tokens == tuple(TEXTS["expected"]["canonical_bpe_tokens"])
assert bpe.encode(TEXTS["surface_variants"]["base"]).ids == bpe.encode(
    TEXTS["surface_variants"]["casing"]
).ids
rare = compare_schemes(TEXTS["rare_strings"]["identifier"])
assert rare["bpe_length"] > rare["word_length"]
assert batch.padding_mask[0][-1] == 0
assert cut.truncated and "invoice" in bpe.decode_pieces(cut.dropped_tokens)
assert defective.silent and not defective.contains("approve_refund", bpe)
assert repaired.contains("approve_refund", bpe)
print("M27 integrity checks passed")
